# Phase 3 — Full-Text Access Assessment via Selenium Word-Count Classification

## Overview
Phase 3 addresses a critical practical barrier: most scientific papers are behind paywalls, so scraping their DOI URLs returns only a login page or brief abstract snippet. This notebook develops a Selenium-based pipeline to load each paper's DOI URL in a headless Chrome browser, count the rendered words, and classify the paper as open access (above threshold) or restricted (below threshold). It also contains earlier attempts to scrape Oxford Academic pages directly, which revealed the need for JavaScript rendering.

---

## Context
The pre-processed dataset (`output_11.csv`) contains 133 papers from the VA Normative Aging Study corpus with DOI URLs and PMIDs. After cleaning, 112 have valid DOI URLs. The open-access subset identified here (`OPEN_ACCESS_SIMPLE_<timestamp>.csv`) becomes the input for Phase 4 extraction.

---

## What This Notebook Does, Step by Step

### Oxford Academic Scraper Tests (Cells 0-3)
Exploratory attempts to scrape a specific Oxford Academic paper directly via requests+BeautifulSoup.

- Cell 0: Fetches an Oxford Academic DOI URL, extracts title and abstract, passes to GPT-3.5-turbo for extraction. Result: empty exposures and outcomes. Oxford Academic requires JavaScript rendering; requests only returns the static HTML shell.

- Cell 1: Tests whether GPT can understand the fetched content at all. Result: Returns API_READ_SUCCESS confirming the API itself is functional. The failure is on the scraping side.

- Cell 2: Adds Oxford-specific CSS selectors (section.abstract, div.abstract) for better extraction. Result: Still returns Not available. JavaScript rendering is required.

- Cell 3: Fetches the first 2000 raw characters and asks GPT to describe what it sees. Result: GPT correctly identifies it is reading a JavaScript-disabled browser warning page, not a research paper. This confirms why all static scraping approaches return nothing useful.

### Selenium Scraper for Full Content (Cell 4)
- Introduces Selenium with headless Chrome to properly render JavaScript-heavy academic pages.
- Uses WebDriverWait to wait for the page body to load, then extracts up to 3000 characters.
- Tests against an Oxford Academic paper (AESurv, PMID 39323093).
- Result: Successfully retrieves 3000 characters including author names, abstract, and introduction. Hits an API version conflict when attempting GPT extraction (module openai has no attribute chat).

### OpenAI Version Fix (Cell 5)
- Pins openai==0.28.1 to resolve the API version conflict caused by newer SDK breaking the .chat attribute syntax.

### Full-Text Extraction with Selenium + GPT (Cell 6)
- Rebuilds the scraper to extract full paper sections from Oxford Academic: abstract, introduction, methods, discussion, conclusion.
- Successfully extracts 15,000 characters from the AESurv paper including a full abstract (1,831 chars) and introduction (4,511 chars).
- Confirms the Selenium + GPT pipeline works for open-access papers.

### Journal Access Checker v1 (Cell 7)
- Loads 133 papers, offers quick test (5 papers) or full run.
- Uses requests.get with browser headers to fetch each DOI URL.
- Result on quick test: Fails because some DOI URLs are NaN.

### Journal Access Checker v2 (Cell 8)
- Adds NaN filtering: removes 21 rows with missing or invalid DOI URLs, leaving 112 valid papers.
- Uses requests.Session with full browser headers for better compatibility.
- Runs quick test on 5 papers successfully, retrieving journal names from live URLs.

### Open Access Classifier via requests (Cell 9)
- Classifies papers as OPEN_ACCESS or RESTRICTED based on content characteristics.
- Runs full analysis on all 112 papers.
- Limitation: Static requests scraping fails to render JavaScript, making results unreliable.

### Selenium Word-Count Classifier - Quick Test (Cell 11)
- Replaces requests with headless Selenium for proper JavaScript rendering.
- Defines simple_word_count(): loads each DOI URL in Chrome, waits for body load, extracts all page text, filters out short tokens, counts remaining words.
- Configurable word threshold (this run: 500 words). Papers above threshold classified OPEN_ACCESS, below classified RESTRICTED.
- Sample result: PMID 24684821 (Chemerin levels paper) returns 3,068 raw words from metabolismjournal.com -- clearly open access.

### Selenium Word-Count Classifier - Full Run (Cell 12)
- Scales to all 112 papers with threshold raised to 1,500 words to reduce false positives from paywall pages that include navigation menus.
- Processes papers sequentially with one headless Chrome instance per paper.
- Saves all results to simple_word_check_<timestamp>.csv.
- Saves open-access subset to OPEN_ACCESS_SIMPLE_<timestamp>.csv -- this is the input for Phase 4.

---

## Key Outputs
- simple_word_check_<timestamp>.csv: word counts and access classification for 112 papers
- OPEN_ACCESS_SIMPLE_<timestamp>.csv: open-access subset used as Phase 4 input

## Why Word Count Works
A paywall or abstract-only page typically renders fewer than 500 words. A full article renders 2,000-15,000+ words. The configurable threshold (500-1,500) correctly classifies open-access papers without requiring publisher-specific logic. It is not perfect -- some institutional-login papers may be misclassified -- but it is fast, generalizable, and requires no publisher-specific knowledge.

## Dependencies
```
selenium (Chrome + ChromeDriver required), beautifulsoup4, requests, pandas, openai, python-dotenv
```
Full analysis of 112 papers takes 30-60 minutes due to browser load time per paper.

In [12]:
import requests
from bs4 import BeautifulSoup
import openai
from dotenv import load_dotenv
import os

# Load API key
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE" 

def get_oup_paper_content(url):
    """Get the main content from an Oxford Academic paper"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=30)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Get title
        title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "No title found"
        
        # Get abstract
        abstract = soup.find('section', class_='abstract')
        abstract_text = abstract.get_text(strip=True) if abstract else "No abstract found"
        
        # Get main content 
        content = soup.find('div', class_='article-body') or soup.find('article')
        content_text = content.get_text(strip=True) if content else "No content found"
        
        return f"TITLE: {title}\n\nABSTRACT: {abstract_text}\n\nCONTENT: {content_text[:5000]}"
        
    except Exception as e:
        return f"Error: {str(e)}"

def extract_elements_simple(content):
    """Simple extraction with ChatGPT"""
    prompt = f"""
    Read this research paper and extract ONLY:
    
    1. EXPOSURES (what was measured as causes):
    - List 2-3 main exposure variables
    
    2. OUTCOMES (what was measured as effects):
    - List 2-3 main outcome variables
    
    Return simple JSON:
    {{
        "exposures": ["exposure1", "exposure2"],
        "outcomes": ["outcome1", "outcome2"]
    }}
    
    Paper content: {content}
    """
    
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"GPT error: {str(e)}"

# Test with Oxford URL
test_url = "https://academic.oup.com/bib/article-lookup/doi/10.1093/bib/bbae479"

print("Fetching paper content...")
paper_content = get_oup_paper_content(test_url)

print("Extracting exposures/outcomes...")
result = extract_elements_simple(paper_content)

print("\n" + "="*50)
print("RESULTS:")
print("="*50)
print(result)

Fetching paper content...
Extracting exposures/outcomes...

RESULTS:
{
    "exposures": [],
    "outcomes": []
}


In [14]:
import requests
from bs4 import BeautifulSoup
import openai
from dotenv import load_dotenv
import os

# Load API key 
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")  or "YOUR_OPENAI_API_KEY_HERE" 


def get_article_content(url):
    """Get basic article content"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=30)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Get title
        title = soup.find('h1').get_text(strip=True) if soup.find('h1') else "No title found"
        
        # Get abstract 
        abstract = soup.find('section', class_='abstract') or soup.find('div', class_='abstract')
        abstract_text = abstract.get_text(strip=True)[:500] if abstract else "No abstract found"
        
        return f"Title: {title}\n\nAbstract: {abstract_text}"
        
    except Exception as e:
        return f"Error fetching content: {str(e)}"

def test_api_understanding(content):
    """Test if API can understand the article content"""
    try:
        # Simple test prompt
        prompt = f"""
        Just look at this research paper content and tell me:
        1. What is the main topic?
        2. Can you identify any exposures or outcomes mentioned?
        
        If you can read this, respond with "API_READ_SUCCESS" and a brief summary.
        
        Content: {content[:1000]}
        """
        
        # This will fail if quota is exceeded, but we'll see the error
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100,
            timeout=10
        )
        
        return response.choices[0].message.content
        
    except Exception as e:
        return f"API_ERROR: {str(e)}"

# Test URL
test_url = "https://academic.oup.com/bib/article-lookup/doi/10.1093/bib/bbae479"

print("🧪 Testing OpenAI API Understanding")
print("=" * 50)

# Get article content
print("1. Fetching article content...")
article_content = get_article_content(test_url)
print("Content fetched successfully")

# Test API understanding
print("2. Testing API understanding...")
api_response = test_api_understanding(article_content)

print("3. API Response:")
print("   " + "=" * 40)
print(f"   {api_response}")
print("   " + "=" * 40)

# Interpret results
if "API_READ_SUCCESS" in api_response:
    print("🎉 SUCCESS: API can read and understand the article!")
    print("   The approach will work once billing is fixed.")
elif "quota" in api_response.lower() or "billing" in api_response.lower():
    print(" BILLING ISSUE: API understands but quota is exceeded")
    print("   The technical approach is sound - just need to fix billing.")
elif "Error" in api_response:
    print(" CONNECTION ISSUE: Could not reach API")
    print("   Check internet connection or API key.")
else:
    print(" UNKNOWN RESPONSE: Need to investigate further")
    print(f"   Response: {api_response}")

🧪 Testing OpenAI API Understanding
1. Fetching article content...
Content fetched successfully
2. Testing API understanding...
3. API Response:
   API_READ_SUCCESS This research paper lacks a title and an abstract, making it difficult to identify the main topic or any exposures or outcomes mentioned.
🎉 SUCCESS: API can read and understand the article!
   The approach will work once billing is fixed.


In [18]:
import requests
from bs4 import BeautifulSoup
import openai
from dotenv import load_dotenv
import os

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE" 


def get_better_article_content(url):
    """Better content extraction from Oxford Academic"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=30)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Better title extraction
        title = soup.find('h1', class_='article-title')
        if not title:
            title = soup.find('h1')
        title_text = title.get_text(strip=True) if title else "No title found"
        
        # Better abstract extraction
        abstract_section = soup.find('section', class_='abstract')
        if not abstract_section:
            abstract_section = soup.find('div', class_='abstract')
        abstract_text = abstract_section.get_text(strip=True) if abstract_section else "No abstract found"
        
        # Get some main content
        content_div = soup.find('div', class_='article-body') or soup.find('article')
        content_text = content_div.get_text(strip=True)[:1000] if content_div else "No content found"
        
        return f"TITLE: {title_text}\n\nABSTRACT: {abstract_text}\n\nCONTENT: {content_text}"
        
    except Exception as e:
        return f"Error: {str(e)}"

def test_complete_extraction():
    """Test the complete extraction pipeline"""
    url = "https://academic.oup.com/bib/article-lookup/doi/10.1093/bib/bbae479"
    
    print("🔍 Fetching improved article content...")
    content = get_better_article_content(url)
    print("✅ Content fetched")
    
    print("📝 Testing extraction...")
    prompt = f"""
    Analyze this research paper and extract:
    
    EXPOSURES: [list 2-3 main exposure variables]
    OUTCOMES: [list 2-3 main outcome variables]
    
    Return format:
    Exposures: exposure1, exposure2
    Outcomes: outcome1, outcome2
    
    Paper content: {content}
    """
    
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
            timeout=20
        )
        
        result = response.choices[0].message.content
        print("🎉 EXTRACTION SUCCESSFUL!")
        print("=" * 50)
        print(result)
        print("=" * 50)
        
    except Exception as e:
        print(f"❌ Extraction failed: {e}")

if __name__ == "__main__":
    test_complete_extraction()

🔍 Fetching improved article content...
✅ Content fetched
📝 Testing extraction...
🎉 EXTRACTION SUCCESSFUL!
Exposures: Not available
Outcomes: Not available


In [22]:
import requests
from bs4 import BeautifulSoup
import openai
from dotenv import load_dotenv
import os

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE" 


def get_article_content(url):
    """Get article content"""
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers, timeout=30)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Get any text content
    all_text = soup.get_text(strip=True)
    return all_text[:2000]  # First 2000 characters

def test_simple_reading():
    """Super simple test: Can the API read and summarize?"""
    url = "https://academic.oup.com/bib/article-lookup/doi/10.1093/bib/bbae479"
    
    print("🔍 Getting article text...")
    content = get_article_content(url)
    print(f"✅ Got {len(content)} characters of text")
    
    print("📖 Asking API to simply describe what it sees...")
    
    prompt = f"""
    Just look at this text and tell me: 
    1. Does this look like a research paper?
    2. What is the main subject or topic?
    3. What type of study is it? (e.g., review, experimental, observational)
    
    Text: {content[:1500]}
    """
    
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200,
            timeout=15
        )
        
        result = response.choices[0].message.content
        print("=" * 60)
        print("📋 API RESPONSE:")
        print("=" * 60)
        print(result)
        print("=" * 60)
        
        # Check if API actually read something
        if "research" in result.lower() or "study" in result.lower():
            print("✅ SUCCESS: API can read and understand the paper content!")
        else:
            print("🤔 API responded but content may not be accessible")
            
    except Exception as e:
        print(f"❌ API Error: {e}")

# Run the test
test_simple_reading()

🔍 Getting article text...
✅ Got 57 characters of text
📖 Asking API to simply describe what it sees...
📋 API RESPONSE:
1. No, this does not look like a research paper.
2. The main subject or topic is troubleshooting internet browser settings.
3. This is not a study, it is simply instructions for enabling JavaScript and cookies on a browser.
✅ SUCCESS: API can read and understand the paper content!


In [24]:
import requests
from bs4 import BeautifulSoup
import openai
from dotenv import load_dotenv
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE" 

def get_article_content_selenium(url):
    """Get article content using Selenium to handle JavaScript"""
    # Configure Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Run in background
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url)
        
        # Wait for page to load
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        # Get page source after JavaScript execution
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        driver.quit()
        
        # Try to find article content in common academic paper selectors
        content_selectors = [
            'div.article-body',
            'div.content',
            'div.fulltext-view',
            'section.abstract',
            'div.abstract',
            'main',
            'article'
        ]
        
        content = ""
        for selector in content_selectors:
            elements = soup.select(selector)
            if elements:
                content = ' '.join([elem.get_text(strip=True) for elem in elements])
                if len(content) > 200:  # Found substantial content
                    break
        
        # Fallback: get all text but try to filter out navigation
        if len(content) < 200:
            # Remove common non-content elements
            for unwanted in soup(["script", "style", "nav", "header", "footer", "aside"]):
                unwanted.decompose()
            content = soup.get_text(strip=True)
        
        return content[:3000]  # First 3000 characters
        
    except Exception as e:
        print(f"Selenium error: {e}")
        return None

def get_article_content_requests(url):
    """Fallback method using requests with better headers"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1'
    }
    
    try:
        session = requests.Session()
        response = session.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Try specific academic content selectors
        content_selectors = [
            'div.article-body',
            'div.content',
            'div.fulltext-view',
            'section.abstract',
            'div.abstract'
        ]
        
        content = ""
        for selector in content_selectors:
            elements = soup.select(selector)
            if elements:
                content = ' '.join([elem.get_text(strip=True) for elem in elements])
                if len(content) > 200:
                    break
        
        if len(content) < 200:
            # Remove unwanted elements
            for unwanted in soup(["script", "style", "nav", "header", "footer", "aside"]):
                unwanted.decompose()
            content = soup.get_text(strip=True)
        
        return content[:3000]
        
    except Exception as e:
        print(f"Requests error: {e}")
        return None

def test_improved_reading():
    """Test with improved content extraction"""
    url = "https://academic.oup.com/bib/article-lookup/doi/10.1093/bib/bbae479"
    
    print("🔍 Getting article text (trying multiple methods)...")
    
    # Try Selenium first (better for JS-heavy sites)
    content = get_article_content_selenium(url)
    method_used = "Selenium"
    
    # Fallback to requests if Selenium fails
    if not content or len(content) < 100:
        print("   Selenium failed, trying requests...")
        content = get_article_content_requests(url)
        method_used = "Requests"
    
    if not content:
        print("❌ Could not retrieve content with either method")
        return
    
    print(f"✅ Got {len(content)} characters using {method_used}")
    print(f"📋 First 200 characters: {content[:200]}...")
    
    # Check if we got the JS warning
    if "javascript" in content.lower() and "enable" in content.lower():
        print("⚠️  Still getting JavaScript warning - site may require authentication")
        print("💡 Try using a different URL or accessing through institutional login")
        return
    
    print("📖 Asking API to analyze content...")
    
    prompt = f"""
    Analyze this academic content and tell me: 
    1. Does this look like a research paper or academic content?
    2. What is the main subject or topic?
    3. What type of study or content is it?
    4. Can you identify any key findings or main points?
    
    Content: {content[:2000]}
    """
    
    try:
        # Updated API call for newer OpenAI versions
        response = openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300,
            timeout=15
        )
        
        result = response.choices[0].message.content
        print("=" * 60)
        print("📋 API RESPONSE:")
        print("=" * 60)
        print(result)
        print("=" * 60)
        
    except Exception as e:
        print(f"❌ API Error: {e}")
        print("💡 Make sure your OpenAI API key is valid and you have credits")

# Alternative: Test with a more accessible paper
def test_with_accessible_paper():
    """Test with a paper that's more likely to be accessible"""
    # Try arXiv paper (usually more accessible)
    urls_to_try = [
        "https://arxiv.org/abs/2301.07041",  # Example arXiv paper
        "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8760467/",  # PubMed Central
    ]
    
    for url in urls_to_try:
        print(f"\n🔍 Trying {url}...")
        content = get_article_content_requests(url)
        
        if content and len(content) > 500:
            print(f"✅ Successfully got {len(content)} characters")
            print(f"📋 Sample: {content[:200]}...")
            break
        else:
            print("❌ No substantial content found")

if __name__ == "__main__":
    # Run improved test
    test_improved_reading()
    
    # Also test with more accessible papers
    print("\n" + "="*60)
    print("TRYING MORE ACCESSIBLE PAPERS:")
    print("="*60)
    test_with_accessible_paper()

🔍 Getting article text (trying multiple methods)...
✅ Got 3000 characters using Selenium
📋 First 200 characters: PDFViewsArticle contentsFigures & tablesSupplementary DataCiteCiteYike Shen, Arce Domingo-Relloso, Allison Kupsco, Marianthi-Anna Kioumourtzoglou, Maria Tellez-Plaza, Jason G Umans, Amanda M Fretts, Y...
📖 Asking API to analyze content...
❌ API Error: module 'openai' has no attribute 'chat'
💡 Make sure your OpenAI API key is valid and you have credits

TRYING MORE ACCESSIBLE PAPERS:

🔍 Trying https://arxiv.org/abs/2301.07041...
✅ Successfully got 3000 characters
📋 Sample: [2301.07041] Verifiable Fully Homomorphic EncryptionComputer Science > Cryptography and SecurityarXiv:2301.07041(cs)[Submitted on 17 Jan 2023 (v1), last revised 11 Feb 2023 (this version, v2)]Title:Ve...


In [26]:
# For the legacy API (what your code expects):
!pip install openai==0.28.1

# OR for the new API:
!pip install openai>=1.0.0

  Attempting uninstall: openai
    Found existing installation: openai 0.28.0
    Uninstalling openai-0.28.0:
      Successfully uninstalled openai-0.28.0
zsh:1: 1.0.0 not found


In [28]:
import requests
from bs4 import BeautifulSoup
import openai
from dotenv import load_dotenv
import os
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY") or "YOUR_OPENAI_API_KEY_HERE" 

def get_full_paper_content(url, max_chars=None):
    """Extract the complete paper content from academic URL"""
    
    # Configure Chrome for paper extraction
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        print(f"🌐 Loading: {url}")
        driver.get(url)
        
        # Wait for content to load
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        # Additional wait for dynamic content
        time.sleep(3)
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        driver.quit()
        
        # Academic paper content extraction strategies
        paper_content = []
        
        print("🔍 Extracting paper sections...")
        
        # Strategy 1: Look for full-text/PDF viewer content
        fulltext_selectors = [
            'div[class*="fulltext"]',
            'div[class*="full-text"]',
            'div[class*="article-body"]',
            'div[class*="content-inner"]',
            'div[class*="paper-content"]',
            'main[class*="content"]',
            'section[class*="body"]'
        ]
        
        for selector in fulltext_selectors:
            elements = soup.select(selector)
            if elements:
                content = ' '.join([elem.get_text(strip=True, separator=' ') for elem in elements])
                if len(content) > 1000:  # Substantial content
                    paper_content.append(("Full text section", content))
                    print(f"✅ Found full text section: {len(content)} chars")
                    break
        
        # Strategy 2: Extract structured paper sections
        sections_found = []
        
        # Abstract
        abstract_selectors = [
            'section[class*="abstract"]',
            'div[class*="abstract"]',
            'div[id*="abstract"]',
            '.abstract',
            'div[data-testid="abstract"]'
        ]
        
        for selector in abstract_selectors:
            elements = soup.select(selector)
            if elements:
                abstract_text = ' '.join([elem.get_text(strip=True, separator=' ') for elem in elements])
                if len(abstract_text) > 100:
                    paper_content.append(("Abstract", abstract_text))
                    sections_found.append("Abstract")
                    print(f"✅ Found abstract: {len(abstract_text)} chars")
                    break
        
        # Introduction, Methods, Results, Discussion, Conclusion
        section_patterns = [
            (r'introduction|intro', 'Introduction'),
            (r'methods|methodology|materials', 'Methods'),
            (r'results|findings', 'Results'),
            (r'discussion|analysis', 'Discussion'),
            (r'conclusion|conclusions|summary', 'Conclusion')
        ]
        
        # Look for section headings and following content
        headings = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
        
        for heading in headings:
            heading_text = heading.get_text(strip=True).lower()
            
            for pattern, section_name in section_patterns:
                if re.search(pattern, heading_text) and section_name not in sections_found:
                    # Get content following this heading
                    section_content = []
                    current = heading.next_sibling
                    
                    while current and len(section_content) < 5:  # Get next few elements
                        if hasattr(current, 'get_text'):
                            text = current.get_text(strip=True, separator=' ')
                            if len(text) > 50:  # Skip very short elements
                                section_content.append(text)
                        elif hasattr(current, 'strip'):  # Text node
                            text = current.strip()
                            if len(text) > 50:
                                section_content.append(text)
                        current = current.next_sibling
                        
                        # Stop at next heading
                        if current and current.name in ['h1', 'h2', 'h3', 'h4', 'h5', 'h6']:
                            break
                    
                    if section_content:
                        combined_content = ' '.join(section_content)
                        paper_content.append((section_name, combined_content))
                        sections_found.append(section_name)
                        print(f"✅ Found {section_name}: {len(combined_content)} chars")
                    break
        
        # Strategy 3: If structured extraction failed, get all meaningful content
        if not paper_content or sum(len(content[1]) for content in paper_content) < 2000:
            print("📄 Falling back to full content extraction...")
            
            # Remove non-content elements
            for unwanted in soup(["script", "style", "nav", "header", "footer", 
                                 "aside", "advertisement", "banner", "navigation"]):
                unwanted.decompose()
            
            # Remove elements with navigation/UI classes
            ui_classes = ['nav', 'menu', 'sidebar', 'footer', 'header', 'banner', 
                         'advertisement', 'social', 'share', 'related', 'recommended']
            
            for class_name in ui_classes:
                for elem in soup.find_all(class_=lambda x: x and any(ui_class in str(x).lower() for ui_class in ui_classes)):
                    elem.decompose()
            
            # Get remaining text content
            main_content = soup.get_text(strip=True, separator=' ')
            
            # Clean up the text
            main_content = re.sub(r'\s+', ' ', main_content)  # Normalize whitespace
            main_content = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)\"\']+', '', main_content)  # Basic cleanup
            
            if len(main_content) > 1000:
                paper_content = [("Full content", main_content)]
                print(f"✅ Extracted full content: {len(main_content)} chars")
        
        # Combine all content
        if paper_content:
            # Sort sections in logical order
            section_order = ['Abstract', 'Introduction', 'Methods', 'Results', 'Discussion', 'Conclusion']
            sorted_content = []
            
            # Add ordered sections first
            for section in section_order:
                for title, content in paper_content:
                    if title == section:
                        sorted_content.append(f"\n=== {title} ===\n{content}")
            
            # Add remaining sections
            for title, content in paper_content:
                if title not in section_order:
                    sorted_content.append(f"\n=== {title} ===\n{content}")
            
            final_content = '\n'.join(sorted_content)
            
            # Apply character limit if specified
            if max_chars:
                final_content = final_content[:max_chars]
            
            return final_content
        
        return None
        
    except Exception as e:
        print(f"❌ Error extracting content: {e}")
        return None

def analyze_paper_with_api(content, custom_prompt=None):
    """Send the extracted paper content to OpenAI for analysis"""
    
    if not content:
        print("❌ No content to analyze")
        return
    
    # Default comprehensive analysis prompt
    if not custom_prompt:
        prompt = f"""
        Please provide a comprehensive analysis of this academic paper:

        1. **Paper Type & Field**: What type of study is this and what field does it belong to?
        
        2. **Main Research Question**: What is the primary research question or hypothesis?
        
        3. **Methodology**: Briefly describe the research methods used.
        
        4. **Key Findings**: What are the most important results or discoveries?
        
        5. **Significance**: Why is this research important or impactful?
        
        6. **Limitations**: Are there any notable limitations mentioned?
        
# 0. create the dictionary from the excel sheet and feed to the api
# Exposure: prompts 1
#  Biomakers 2/ Health outcomes 3: prompts
# Samples size 4 / population studied 4:prompts
# Models used 5
# Associations & Stats significance 6: prompts
# Be concise in the summary, (just one sentence) 7 
# Macth the accuracy witht the excel table: next step
# Open source text 
# Use different promts: later
# Ask Dhru how to fecth the papers links
# Put the outcomes in the layer 2, in a dictionary, and ask gpt to match what it found to those terms. If something is not related, it can create new terms that match the findings. 
# Don't tell gpt what is the outcomes or the exposure
# Create an anthology

        Paper Content:
        {content[:8000]}  # Use first 8000 chars to stay within token limits
        """
    else:
        prompt = f"{custom_prompt}\n\nPaper Content:\n{content[:8000]}"
    
    try:
        print("🤖 Sending to OpenAI for analysis...")
        
        # Handle both old and new OpenAI API versions
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1000,
            temperature=0.3
        )
        
        result = response.choices[0].message.content
        
        print("\n" + "="*80)
        print("📊 PAPER ANALYSIS")
        print("="*80)
        print(result)
        print("="*80)
        
        return result
        
    except Exception as e:
        print(f"❌ OpenAI API Error: {e}")
        print("💡 Check your API key and make sure you have sufficient credits")
        return None

def main():
    """Main function to extract and analyze a paper"""
    
    # Your original URL
    url = "https://academic.oup.com/bib/article-lookup/doi/10.1093/bib/bbae479"
    
    print("🚀 Starting full paper extraction...")
    print(f"📄 Target URL: {url}")
    
    # Extract the full paper content
    content = get_full_paper_content(url, max_chars=15000)  # Limit for API efficiency
    
    if content:
        print(f"\n✅ Successfully extracted {len(content)} characters")
        print(f"📋 Content preview:\n{content[:500]}...\n")
        
        # Analyze with OpenAI
        analysis = analyze_paper_with_api(content)
        
        # Optional: Save content to file for later use
        with open("extracted_paper.txt", "w", encoding="utf-8") as f:
            f.write(content)
        print("💾 Full content saved to 'extracted_paper.txt'")
        
    else:
        print("❌ Failed to extract paper content")
        print("💡 Try a different URL or check if the paper is behind a paywall")

if __name__ == "__main__":
    main()

🚀 Starting full paper extraction...
📄 Target URL: https://academic.oup.com/bib/article-lookup/doi/10.1093/bib/bbae479
🌐 Loading: https://academic.oup.com/bib/article-lookup/doi/10.1093/bib/bbae479
🔍 Extracting paper sections...
✅ Found full text section: 50915 chars
✅ Found abstract: 1831 chars
✅ Found Introduction: 4511 chars
✅ Found Methods: 731 chars
✅ Found Discussion: 3338 chars
✅ Found Conclusion: 877 chars

✅ Successfully extracted 15000 characters
📋 Content preview:

=== Abstract ===
Coronary heart disease (CHD) is one of the leading causes of mortality and morbidity in the United States. Accurate time-to-event CHD prediction models with high-dimensional DNA methylation and clinical features may assist with early prediction and intervention strategies. We developed a state-of-the-art deep learning autoencoder survival analysis model (AESurv) to effectively analyze high-dimensional blood DNA methylation features and traditional clinical risk factors by learn...

🤖 Sending to Ope

In [38]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime

def get_journal_from_url(url, timeout=10):
    """
    Simple function to get journal name from a DOI URL
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        print(f"   🌐 Checking: {url}")
        
        response = requests.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Try multiple strategies to find journal name
        journal_name = None
        
        # Strategy 1: Look for common journal name selectors
        selectors = [
            'meta[name="citation_journal_title"]',
            'meta[property="citation_journal_title"]',
            'meta[name="dc.source"]',
            '.journal-title',
            '.journalTitle',
            '.source-title',
            '[data-testid="journal-title"]'
        ]
        
        for selector in selectors:
            element = soup.select_one(selector)
            if element:
                journal_name = element.get('content') or element.get_text(strip=True)
                if journal_name:
                    break
        
        # Strategy 2: Look for journal info in page title or headers
        if not journal_name:
            title_tag = soup.find('title')
            if title_tag:
                title_text = title_tag.get_text()
                # Common patterns in titles
                if ' | ' in title_text:
                    parts = title_text.split(' | ')
                    # Usually journal name is at the end
                    journal_name = parts[-1].strip()
                elif ' - ' in title_text:
                    parts = title_text.split(' - ')
                    journal_name = parts[-1].strip()
        
        # Strategy 3: Look in breadcrumbs or navigation
        if not journal_name:
            breadcrumb_selectors = ['.breadcrumb', '.navigation', '.journal-nav']
            for selector in breadcrumb_selectors:
                element = soup.select_one(selector)
                if element:
                    journal_name = element.get_text(strip=True)
                    break
        
        # Clean up journal name
        if journal_name:
            journal_name = journal_name.replace('\n', ' ').replace('\t', ' ')
            journal_name = ' '.join(journal_name.split())  # Remove extra whitespace
            
            # Remove common suffixes that aren't part of journal name
            suffixes_to_remove = [
                '- PubMed', '| PubMed', 'PubMed',
                '- PMC', '| PMC', 'PMC',
                'Home', 'Homepage'
            ]
            
            for suffix in suffixes_to_remove:
                if journal_name.endswith(suffix):
                    journal_name = journal_name[:-len(suffix)].strip()
        
        return journal_name or "Journal not found"
        
    except requests.exceptions.Timeout:
        return "Timeout"
    except requests.exceptions.RequestException as e:
        return f"Error: {str(e)}"
    except Exception as e:
        return f"Parse error: {str(e)}"

def check_paper_access(csv_file, sample_size=None, delay=1):
    """
    Check if we can access papers and get journal names
    
    Args:
        csv_file: Path to CSV file with paper data
        sample_size: If specified, only test this many papers (for quick testing)
        delay: Seconds to wait between requests (be respectful to servers)
    """
    
    print("📊 Loading paper dataset...")
    
    # Load CSV file
    try:
        df = pd.read_csv(csv_file)
        print(f"✅ Loaded {len(df)} papers from CSV")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return
    
    # Check required columns
    required_columns = ['PMID', 'Title', 'DOI_URL']
    missing_columns = [col for col in required_columns if col not in df.columns]
    
    if missing_columns:
        print(f"❌ Missing required columns: {missing_columns}")
        print(f"Available columns: {list(df.columns)}")
        return
    
    # Sample if requested
    if sample_size and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=42)
        print(f"🎯 Testing with sample of {len(df)} papers")
    
    # Add results columns
    df['Journal'] = None
    df['Access_Status'] = None
    df['Check_Time'] = None
    
    print(f"\n🚀 Starting journal access check...")
    print(f"⏰ Estimated time: {len(df) * (delay + 2)} seconds")
    print("=" * 60)
    
    # Check each paper
    for index, row in df.iterrows():
        pmid = row['PMID']
        title = row['Title']
        doi_url = row['DOI_URL']
        
        print(f"\n📄 Paper {index + 1}/{len(df)}: PMID {pmid}")
        print(f"📋 {title[:80]}...")
        
        # Get journal info
        start_time = time.time()
        journal = get_journal_from_url(doi_url)
        check_time = time.time() - start_time
        
        # Update dataframe
        df.at[index, 'Journal'] = journal
        df.at[index, 'Check_Time'] = f"{check_time:.2f}s"
        
        # Determine access status
        if journal and "not found" not in journal.lower() and "error" not in journal.lower() and "timeout" not in journal.lower():
            df.at[index, 'Access_Status'] = "✅ Success"
            print(f"   ✅ Journal: {journal}")
        else:
            df.at[index, 'Access_Status'] = "❌ Failed"
            print(f"   ❌ {journal}")
        
        # Be respectful to servers
        if delay > 0:
            time.sleep(delay)
    
    # Generate summary
    print("\n" + "=" * 60)
    print("📊 SUMMARY RESULTS")
    print("=" * 60)
    
    success_count = len(df[df['Access_Status'] == '✅ Success'])
    total_count = len(df)
    success_rate = (success_count / total_count) * 100
    
    print(f"Total papers checked: {total_count}")
    print(f"Successfully accessed: {success_count}")
    print(f"Failed to access: {total_count - success_count}")
    print(f"Success rate: {success_rate:.1f}%")
    
    # Show journal breakdown
    print(f"\n📚 JOURNALS FOUND:")
    journal_counts = df[df['Access_Status'] == '✅ Success']['Journal'].value_counts()
    for journal, count in journal_counts.head(10).items():
        print(f"   • {journal}: {count} papers")
    
    if len(journal_counts) > 10:
        print(f"   ... and {len(journal_counts) - 10} more journals")
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"journal_check_results_{timestamp}.csv"
    df.to_csv(output_file, index=False)
    
    print(f"\n💾 Results saved to: {output_file}")
    print(f"🎉 Journal access check complete!")
    
    return df

# Quick test function
def quick_test(csv_file):
    """Quick test with just 5 papers to see if it works"""
    print("🧪 Running quick test with 5 papers...")
    return check_paper_access(csv_file, sample_size=5, delay=1)

# Full check function  
def full_check(csv_file):
    """Check all papers (will take longer)"""
    print("🚀 Running full check on all papers...")
    response = input("This will check all papers and may take a while. Continue? (y/n): ")
    if response.lower() == 'y':
        return check_paper_access(csv_file, delay=2)  # Slightly longer delay for full check
    else:
        print("👍 Check cancelled.")
        return None

if __name__ == "__main__":
    csv_file = "output_11.csv"
    
    print("VA Normative Aging Study - Journal Access Checker")
    print("=" * 50)
    
    choice = input("\nChoose:\n1. Quick test (5 papers)\n2. Full check (all papers)\nEnter 1 or 2: ")
    
    if choice == "1":
        results = quick_test(csv_file)
    elif choice == "2":
        results = full_check(csv_file)
    else:
        print("Invalid choice. Running quick test...")
        results = quick_test(csv_file)

VA Normative Aging Study - Journal Access Checker



Choose:
1. Quick test (5 papers)
2. Full check (all papers)
Enter 1 or 2:  1


🧪 Running quick test with 5 papers...
📊 Loading paper dataset...
✅ Loaded 133 papers from CSV
🎯 Testing with sample of 5 papers

🚀 Starting journal access check...
⏰ Estimated time: 15 seconds

📄 Paper 52/5: PMID nan
📋 DNA methylation of telomere-related genes and cancer risk...
   🌐 Checking: nan
   ❌ Error: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?

📄 Paper 70/5: PMID nan
📋 Quantile regression analysis of the distributional effects of air pollution on b...
   🌐 Checking: nan
   ❌ Error: Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?

📄 Paper 32/5: PMID 31636367.0
📋 Blood DNA methylation biomarkers of cumulative lead exposure in adults....
   🌐 Checking: https://doi.org/10.1038/s41370-019-0183-9
   ✅ Journal: Journal of Exposure Science & Environmental Epidemiology

📄 Paper 43/5: PMID 30102601.0
📋 Bone Lead Levels and Risk of Incident Primary Open-Angle Glaucoma: The VA Normat...
   🌐 Checking: https://doi.org/10.1289/EHP3442
   ❌ Err

In [42]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime

def get_journal_from_url(url, timeout=15):
    """
    Simple function to get journal name from a DOI URL
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive'
    }
    
    try:
        print(f"   🌐 Checking: {url}")
        
        # Add session for better handling
        session = requests.Session()
        session.headers.update(headers)
        
        response = session.get(url, timeout=timeout, allow_redirects=True)
        
        # Handle different response codes
        if response.status_code == 403:
            return "Access Forbidden (403)"
        elif response.status_code == 404:
            return "Page Not Found (404)"
        elif response.status_code != 200:
            return f"HTTP Error ({response.status_code})"
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Try multiple strategies to find journal name
        journal_name = None
        
        # Strategy 1: Look for common journal name selectors (more comprehensive)
        selectors = [
            'meta[name="citation_journal_title"]',
            'meta[property="citation_journal_title"]',
            'meta[name="dc.source"]',
            'meta[name="prism.publicationName"]',
            '.journal-title',
            '.journalTitle',
            '.source-title',
            '[data-testid="journal-title"]',
            '.publication-title',
            '.journal-name',
            'span.journal-title',
            'h1.journal-title'
        ]
        
        for selector in selectors:
            element = soup.select_one(selector)
            if element:
                journal_name = element.get('content') or element.get_text(strip=True)
                if journal_name and len(journal_name.strip()) > 0:
                    break
        
        # Strategy 2: Look for journal info in page title or headers
        if not journal_name:
            title_tag = soup.find('title')
            if title_tag:
                title_text = title_tag.get_text()
                # Common patterns in titles
                if ' | ' in title_text:
                    parts = title_text.split(' | ')
                    # Usually journal name is at the end or second to last
                    for part in reversed(parts):
                        if len(part.strip()) > 3 and 'doi' not in part.lower():
                            journal_name = part.strip()
                            break
                elif ' - ' in title_text:
                    parts = title_text.split(' - ')
                    for part in reversed(parts):
                        if len(part.strip()) > 3 and 'doi' not in part.lower():
                            journal_name = part.strip()
                            break
        
        # Clean up journal name
        if journal_name:
            journal_name = journal_name.replace('\n', ' ').replace('\t', ' ')
            journal_name = ' '.join(journal_name.split())  # Remove extra whitespace
            
            # Remove common suffixes that aren't part of journal name
            suffixes_to_remove = [
                '- PubMed', '| PubMed', 'PubMed',
                '- PMC', '| PMC', 'PMC',
                'Home', 'Homepage', '- Home',
                'ScienceDirect', 'Wiley Online Library',
                'SpringerLink'
            ]
            
            for suffix in suffixes_to_remove:
                if journal_name.endswith(suffix):
                    journal_name = journal_name[:-len(suffix)].strip()
            
            # Remove leading/trailing punctuation
            journal_name = journal_name.strip('.,|-: ')
        
        return journal_name or "Journal not found"
        
    except requests.exceptions.Timeout:
        return "Timeout"
    except requests.exceptions.ConnectionError:
        return "Connection Error"
    except requests.exceptions.RequestException as e:
        return f"Request Error: {str(e)[:50]}"
    except Exception as e:
        return f"Parse error: {str(e)[:50]}"

def check_paper_access(csv_file, sample_size=None, delay=1):
    """
    Check if we can access papers and get journal names
    
    Args:
        csv_file: Path to CSV file with paper data
        sample_size: If specified, only test this many papers (for quick testing)
        delay: Seconds to wait between requests (be respectful to servers)
    """
    
    print("📊 Loading paper dataset...")
    
    # Load CSV file
    try:
        df = pd.read_csv(csv_file)
        print(f"✅ Loaded {len(df)} papers from CSV")
        
        # Clean the data - remove rows with missing DOI URLs
        original_count = len(df)
        
        # Remove rows with NaN in DOI_URL column
        df = df.dropna(subset=['DOI_URL'])
        
        # Convert DOI_URL to string and filter valid URLs
        df['DOI_URL'] = df['DOI_URL'].astype(str)
        df = df[df['DOI_URL'] != 'nan']
        df = df[df['DOI_URL'].str.startswith('http')]
        
        # Also clean PMID column
        df = df.dropna(subset=['PMID'])
        
        if len(df) < original_count:
            print(f"🧹 Cleaned dataset: removed {original_count - len(df)} rows with missing/invalid data")
            print(f"📊 Final dataset: {len(df)} papers with valid DOI URLs")
        
        if len(df) == 0:
            print("❌ No valid papers found after cleaning!")
            return
        
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return
    
    # Check required columns
    required_columns = ['PMID', 'Title', 'DOI_URL']
    missing_columns = [col for col in required_columns if col not in df.columns]
    
    if missing_columns:
        print(f"❌ Missing required columns: {missing_columns}")
        print(f"Available columns: {list(df.columns)}")
        return
    
    # Sample if requested
    if sample_size and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        print(f"🎯 Testing with sample of {len(df)} papers")
    
    # Add results columns
    df['Journal'] = None
    df['Access_Status'] = None
    df['Check_Time'] = None
    
    print(f"\n🚀 Starting journal access check...")
    print(f"⏰ Estimated time: {len(df) * (delay + 2)} seconds")
    print("=" * 60)
    
    # Check each paper
    for index, row in df.iterrows():
        pmid = row['PMID']
        title = row['Title']
        doi_url = row['DOI_URL']
        
        print(f"\n📄 Paper {index + 1}/{len(df)}: PMID {pmid}")
        print(f"📋 {str(title)[:80]}...")
        
        # Get journal info
        start_time = time.time()
        journal = get_journal_from_url(doi_url)
        check_time = time.time() - start_time
        
        # Update dataframe
        df.at[index, 'Journal'] = journal
        df.at[index, 'Check_Time'] = f"{check_time:.2f}s"
        
        # Determine access status (more specific criteria)
        if (journal and 
            "not found" not in journal.lower() and 
            "error" not in journal.lower() and 
            "timeout" not in journal.lower() and
            "forbidden" not in journal.lower() and
            "http error" not in journal.lower() and
            len(journal.strip()) > 3):
            df.at[index, 'Access_Status'] = "✅ Success"
            print(f"   ✅ Journal: {journal}")
        else:
            df.at[index, 'Access_Status'] = "❌ Failed"
            print(f"   ❌ {journal}")
        
        # Be respectful to servers
        if delay > 0:
            time.sleep(delay)
    
    # Generate summary
    print("\n" + "=" * 60)
    print("📊 SUMMARY RESULTS")
    print("=" * 60)
    
    success_count = len(df[df['Access_Status'] == '✅ Success'])
    total_count = len(df)
    success_rate = (success_count / total_count) * 100
    
    print(f"Total papers checked: {total_count}")
    print(f"Successfully accessed: {success_count}")
    print(f"Failed to access: {total_count - success_count}")
    print(f"Success rate: {success_rate:.1f}%")
    
    # Show journal breakdown
    if success_count > 0:
        print(f"\n📚 JOURNALS FOUND:")
        journal_counts = df[df['Access_Status'] == '✅ Success']['Journal'].value_counts()
        for journal, count in journal_counts.head(10).items():
            print(f"   • {journal}: {count} papers")
        
        if len(journal_counts) > 10:
            print(f"   ... and {len(journal_counts) - 10} more journals")
    else:
        print(f"\n⚠️ No journals successfully extracted")
    
    # Show failure reasons
    print(f"\n❌ FAILURE BREAKDOWN:")
    failed_df = df[df['Access_Status'] == '❌ Failed']
    if len(failed_df) > 0:
        failure_reasons = failed_df['Journal'].value_counts()
        for reason, count in failure_reasons.items():
            print(f"   • {reason}: {count} papers")
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = f"journal_check_results_{timestamp}.csv"
    df.to_csv(output_file, index=False)
    
    print(f"\n💾 Results saved to: {output_file}")
    print(f"🎉 Journal access check complete!")
    
    return df

# Quick test function
def quick_test(csv_file):
    """Quick test with just 5 papers to see if it works"""
    print("🧪 Running quick test with 5 papers...")
    return check_paper_access(csv_file, sample_size=5, delay=1)

# Full check function  
def full_check(csv_file):
    """Check all papers (will take longer)"""
    print("🚀 Running full check on all papers...")
    response = input("This will check all papers and may take a while. Continue? (y/n): ")
    if response.lower() == 'y':
        return check_paper_access(csv_file, delay=2)  # Slightly longer delay for full check
    else:
        print("👍 Check cancelled.")
        return None

if __name__ == "__main__":
    # Update this path to your CSV file
    csv_file = "output_11.csv"
    
    print("Cohort Network Knowledge Base - Journal Access Checker")
    print("=" * 60)
    
    choice = input("\nChoose:\n1. Quick test (5 papers)\n2. Full check (all papers)\nEnter 1 or 2: ")
    
    if choice == "1":
        results = quick_test(csv_file)
    elif choice == "2":
        results = full_check(csv_file)
    else:
        print("Invalid choice. Running quick test...")
        results = quick_test(csv_file)

Cohort Network Knowledge Base - Journal Access Checker



Choose:
1. Quick test (5 papers)
2. Full check (all papers)
Enter 1 or 2:  1


🧪 Running quick test with 5 papers...
📊 Loading paper dataset...
✅ Loaded 133 papers from CSV
🧹 Cleaned dataset: removed 21 rows with missing/invalid data
📊 Final dataset: 112 papers with valid DOI URLs
🎯 Testing with sample of 5 papers

🚀 Starting journal access check...
⏰ Estimated time: 15 seconds

📄 Paper 1/5: PMID 24684821.0
📋 Chemerin levels as predictor of acute coronary events: a case-control study nest...
   🌐 Checking: https://doi.org/10.1016/j.metabol.2014.02.013
   ❌ Journal not found

📄 Paper 2/5: PMID 22336131.0
📋 Residential black carbon exposure and circulating markers of systemic inflammati...
   🌐 Checking: https://doi.org/10.1289/ehp.1103982
   ❌ Access Forbidden (403)

📄 Paper 3/5: PMID 25715694.0
📋 Serum tocopherol levels and vitamin E intake are associated with lung function i...
   🌐 Checking: https://doi.org/10.1016/j.clnu.2015.01.020
   ❌ Journal not found

📄 Paper 4/5: PMID 27453791.0
📋 Long-term ambient particle exposures and blood DNA methylation age: findin

In [46]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime

def check_access_status(url, timeout=15):
    """
    Check if a paper is open access or restricted
    Returns: (access_status, details)
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Accept-Encoding': 'gzip, deflate',
        'Connection': 'keep-alive'
    }
    
    try:
        print(f"   🌐 Checking: {url}")
        
        session = requests.Session()
        session.headers.update(headers)
        
        response = session.get(url, timeout=timeout, allow_redirects=True)
        
        # Check HTTP status codes
        if response.status_code == 403:
            return "RESTRICTED", "403 Forbidden - Access denied"
        elif response.status_code == 404:
            return "NOT_FOUND", "404 Not Found - Page doesn't exist"
        elif response.status_code == 401:
            return "RESTRICTED", "401 Unauthorized - Login required"
        elif response.status_code == 429:
            return "RATE_LIMITED", "429 Too Many Requests - Rate limited"
        elif response.status_code != 200:
            return "ERROR", f"HTTP {response.status_code}"
        
        # If we got a 200 response, check the content
        soup = BeautifulSoup(response.text, 'html.parser')
        content_text = soup.get_text().lower()
        
        # Check for content accessibility indicators
        content_length = len(content_text)
        
        # Look for signs that it's restricted/paywall
        restriction_indicators = [
            'login required', 'sign in', 'subscription required',
            'purchase access', 'institutional access', 'paywall',
            'access denied', 'subscription needed', 'premium content',
            'purchase article', 'buy article', 'rent article',
            'this content is restricted', 'members only'
        ]
        
        # Look for open access indicators
        open_access_indicators = [
            'open access', 'free access', 'publicly available',
            'creative commons', 'cc by', 'full text available',
            'pmc free article', 'free pmc article', 'download pdf'
        ]
        
        # Check for restriction signs
        has_restrictions = any(indicator in content_text for indicator in restriction_indicators)
        
        # Check for open access signs
        has_open_access = any(indicator in content_text for indicator in open_access_indicators)
        
        # Look for substantial content (indicates we can access the paper)
        has_substantial_content = content_length > 5000  # Arbitrary threshold
        
        # Look for abstract or full text sections
        content_sections = soup.find_all(['div', 'section'], class_=lambda x: x and any(
            term in str(x).lower() for term in ['abstract', 'fulltext', 'full-text', 'content', 'article-body']
        ))
        
        has_content_sections = len(content_sections) > 0
        
        # Decision logic
        if has_restrictions:
            return "RESTRICTED", "Contains paywall/login indicators"
        elif has_open_access:
            return "OPEN_ACCESS", "Contains open access indicators"
        elif has_substantial_content and has_content_sections:
            return "LIKELY_OPEN", f"Substantial content available ({content_length} chars)"
        elif content_length < 1000:
            return "RESTRICTED", "Very limited content - likely restricted"
        else:
            return "UNKNOWN", f"Moderate content ({content_length} chars) - status unclear"
        
    except requests.exceptions.Timeout:
        return "TIMEOUT", "Request timed out"
    except requests.exceptions.ConnectionError:
        return "CONNECTION_ERROR", "Could not connect to server"
    except requests.exceptions.RequestException as e:
        return "REQUEST_ERROR", f"Request failed: {str(e)[:50]}"
    except Exception as e:
        return "PARSE_ERROR", f"Error parsing response: {str(e)[:50]}"

def check_all_papers_access(csv_file, sample_size=None, delay=2):
    """
    Check access status for all papers and create separate files for open vs restricted
    """
    
    print("📊 Loading paper dataset...")
    
    try:
        df = pd.read_csv(csv_file)
        print(f"✅ Loaded {len(df)} papers from CSV")
        
        # Clean the data
        original_count = len(df)
        df = df.dropna(subset=['DOI_URL', 'PMID'])
        df['DOI_URL'] = df['DOI_URL'].astype(str)
        df = df[df['DOI_URL'] != 'nan']
        df = df[df['DOI_URL'].str.startswith('http')]
        
        if len(df) < original_count:
            print(f"🧹 Cleaned dataset: removed {original_count - len(df)} rows with missing/invalid data")
        
        print(f"📊 Final dataset: {len(df)} papers with valid DOI URLs")
        
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return
    
    # Sample if requested
    if sample_size and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
        print(f"🎯 Testing with sample of {len(df)} papers")
    
    # Add results columns
    df['Access_Status'] = None
    df['Status_Details'] = None
    df['Check_Time'] = None
    
    print(f"\n🚀 Starting access status check...")
    print(f"⏰ Estimated time: {len(df) * (delay + 2)} seconds")
    print("=" * 70)
    
    # Check each paper
    for index, row in df.iterrows():
        pmid = row['PMID']
        title = row['Title']
        doi_url = row['DOI_URL']
        
        print(f"\n📄 Paper {index + 1}/{len(df)}: PMID {pmid}")
        print(f"📋 {str(title)[:80]}...")
        
        # Check access status
        start_time = time.time()
        access_status, details = check_access_status(doi_url)
        check_time = time.time() - start_time
        
        # Update dataframe
        df.at[index, 'Access_Status'] = access_status
        df.at[index, 'Status_Details'] = details
        df.at[index, 'Check_Time'] = f"{check_time:.2f}s"
        
        # Show result with appropriate emoji
        status_emoji = {
            'OPEN_ACCESS': '🟢',
            'LIKELY_OPEN': '🟡', 
            'RESTRICTED': '🔴',
            'NOT_FOUND': '⚫',
            'TIMEOUT': '⏰',
            'ERROR': '❌',
            'UNKNOWN': '❓'
        }
        
        emoji = status_emoji.get(access_status, '❓')
        print(f"   {emoji} {access_status}: {details}")
        
        # Be respectful to servers
        if delay > 0:
            time.sleep(delay)
    
    # Generate summary
    print("\n" + "=" * 70)
    print("📊 ACCESS STATUS SUMMARY")
    print("=" * 70)
    
    status_counts = df['Access_Status'].value_counts()
    total_count = len(df)
    
    for status, count in status_counts.items():
        percentage = (count / total_count) * 100
        emoji = status_emoji.get(status, '❓')
        print(f"{emoji} {status}: {count} papers ({percentage:.1f}%)")
    
    # Create separate datasets
    open_papers = df[df['Access_Status'].isin(['OPEN_ACCESS', 'LIKELY_OPEN'])]
    restricted_papers = df[df['Access_Status'].isin(['RESTRICTED'])]
    
    print(f"\n📈 ACCESSIBLE PAPERS: {len(open_papers)} total")
    print(f"🚫 RESTRICTED PAPERS: {len(restricted_papers)} total")
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save complete results
    complete_file = f"access_check_complete_{timestamp}.csv"
    df.to_csv(complete_file, index=False)
    print(f"\n💾 Complete results saved to: {complete_file}")
    
    # Save open access papers only
    if len(open_papers) > 0:
        open_access_file = f"open_access_papers_{timestamp}.csv"
        open_papers[['PMID', 'Title', 'DOI_URL', 'Access_Status', 'Status_Details']].to_csv(
            open_access_file, index=False
        )
        print(f"🟢 Open access papers saved to: {open_access_file}")
        
        print(f"\n📚 OPEN ACCESS PAPERS PREVIEW:")
        for _, paper in open_papers.head(5).iterrows():
            print(f"   • PMID {paper['PMID']}: {str(paper['Title'])[:60]}...")
    
    # Save restricted papers for reference
    if len(restricted_papers) > 0:
        restricted_file = f"restricted_papers_{timestamp}.csv"
        restricted_papers[['PMID', 'Title', 'DOI_URL', 'Status_Details']].to_csv(
            restricted_file, index=False
        )
        print(f"🔴 Restricted papers saved to: {restricted_file}")
    
    print(f"\n🎉 Access check complete!")
    print(f"✅ You have {len(open_papers)} papers ready for batch processing!")
    
    return df

def quick_access_test(csv_file):
    """Quick test with 10 papers"""
    print("🧪 Running quick access test with 10 papers...")
    return check_all_papers_access(csv_file, sample_size=10, delay=1)

def full_access_check(csv_file):
    """Check all papers"""
    print("🚀 Running full access check on all papers...")
    response = input("This will check all papers and may take 10-20 minutes. Continue? (y/n): ")
    if response.lower() == 'y':
        return check_all_papers_access(csv_file, delay=2)
    else:
        print("👍 Check cancelled.")
        return None

if __name__ == "__main__":
    csv_file = "output_11.csv"
    
    print("Cohort Network Knowledge Base - Open Access Checker")
    print("=" * 60)
    print("🎯 Goal: Identify which papers are openly accessible for batch processing")
    
    choice = input("\nChoose:\n1. Quick test (10 papers)\n2. Full check (all papers)\nEnter 1 or 2: ")
    
    if choice == "1":
        results = quick_access_test(csv_file)
    elif choice == "2":
        results = full_access_check(csv_file)
    else:
        print("Invalid choice. Running quick test...")
        results = quick_access_test(csv_file)

Cohort Network Knowledge Base - Open Access Checker
🎯 Goal: Identify which papers are openly accessible for batch processing



Choose:
1. Quick test (10 papers)
2. Full check (all papers)
Enter 1 or 2:  2


🚀 Running full access check on all papers...


This will check all papers and may take 10-20 minutes. Continue? (y/n):  y


📊 Loading paper dataset...
✅ Loaded 133 papers from CSV
🧹 Cleaned dataset: removed 21 rows with missing/invalid data
📊 Final dataset: 112 papers with valid DOI URLs

🚀 Starting access status check...
⏰ Estimated time: 448 seconds

📄 Paper 1/112: PMID 30414594.0
📋 Accelerated DNA methylation age and the use of antihypertensive medication among...
   🌐 Checking: https://doi.org/10.18632/aging.101626
   🟢 OPEN_ACCESS: Contains open access indicators

📄 Paper 2/112: PMID 28592514.0
📋 A Western Diet Pattern Is Associated with Higher Concentrations of Blood and Bon...
   🌐 Checking: https://doi.org/10.3945/jn.117.249060
   🔴 RESTRICTED: Very limited content - likely restricted

📄 Paper 3/112: PMID 24145859.0
📋 Self-reported sleep and β-amyloid deposition in community-dwelling older adults....
   🌐 Checking: https://doi.org/10.1001/jamaneurol.2013.4258
   🔴 RESTRICTED: 403 Forbidden - Access denied

📄 Paper 4/112: PMID 26791184.0
📋 Dietary anthocyanin intake and age-related decline in lung fu

In [51]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [58]:
import pandas as pd
import time
from datetime import datetime
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

def simple_word_count(url, timeout=20):
    """
    Simple word counting with minimal filtering
    """
    
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = None
    try:
        print(f"   🌐 Loading: {url}")
        
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url)
        
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(3)
        
        final_url = driver.current_url
        print(f"   📍 Final URL: {final_url}")
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Remove only the most basic unwanted elements
        for unwanted in soup(["script", "style", "nav", "header", "footer"]):
            unwanted.decompose()
        
        # Get all text
        full_text = soup.get_text(separator=' ', strip=True)
        
        # Very basic cleaning - just normalize whitespace
        full_text = re.sub(r'\s+', ' ', full_text)
        
        # Simple word split and count
        words = full_text.split()
        
        # Minimal filtering - just remove very short words and pure numbers
        filtered_words = [word for word in words if len(word) >= 2 and not word.isdigit()]
        
        word_count = len(filtered_words)
        
        # Show first few words for verification
        sample = ' '.join(filtered_words[:20])
        
        print(f"   📊 Raw words: {len(words)}, Filtered: {word_count}")
        print(f"   📝 Sample: {sample}...")
        
        return word_count, sample
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")
        return 0, f"Error: {str(e)}"
    
    finally:
        if driver:
            driver.quit()

def simple_check(csv_file, threshold=1000, sample_size=5):
    """
    Simple check with minimal complexity
    """
    
    print("📊 Loading data...")
    df = pd.read_csv(csv_file)
    
    # Basic cleaning
    df = df.dropna(subset=['DOI_URL', 'PMID'])
    df = df[df['DOI_URL'].str.startswith('http')]
    
    if sample_size:
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    print(f"✅ Testing {len(df)} papers")
    print(f"📏 Threshold: {threshold} words")
    print("=" * 60)
    
    results = []
    
    for i, row in df.iterrows():
        print(f"\n📄 Paper {i+1}/{len(df)}: PMID {row['PMID']}")
        print(f"📋 {row['Title'][:80]}...")
        
        word_count, sample = simple_word_count(row['DOI_URL'])
        
        status = "OPEN_ACCESS" if word_count >= threshold else "RESTRICTED"
        emoji = "🟢" if status == "OPEN_ACCESS" else "🔴"
        
        print(f"   {emoji} {status}: {word_count:,} words")
        
        results.append({
            'PMID': row['PMID'],
            'Title': row['Title'],
            'DOI_URL': row['DOI_URL'],
            'Word_Count': word_count,
            'Status': status,
            'Sample': sample
        })
        
        time.sleep(2)
    
    # Summary
    results_df = pd.DataFrame(results)
    open_count = len(results_df[results_df['Status'] == 'OPEN_ACCESS'])
    restricted_count = len(results_df[results_df['Status'] == 'RESTRICTED'])
    
    print("\n" + "=" * 60)
    print("📊 RESULTS SUMMARY")
    print("=" * 60)
    print(f"🟢 OPEN ACCESS: {open_count} papers")
    print(f"🔴 RESTRICTED: {restricted_count} papers")
    
    if open_count > 0:
        print(f"\n📚 OPEN ACCESS PAPERS:")
        open_papers = results_df[results_df['Status'] == 'OPEN_ACCESS'].sort_values('Word_Count', ascending=False)
        for _, paper in open_papers.iterrows():
            print(f"   • {paper['Word_Count']:,} words - PMID {paper['PMID']}")
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_df.to_csv(f"simple_word_check_{timestamp}.csv", index=False)
    
    if open_count > 0:
        open_papers[['PMID', 'Title', 'DOI_URL', 'Word_Count']].to_csv(
            f"OPEN_ACCESS_SIMPLE_{timestamp}.csv", index=False
        )
        print(f"\n💾 Open access papers saved to: OPEN_ACCESS_SIMPLE_{timestamp}.csv")
    
    return results_df

if __name__ == "__main__":
    csv_file = "output_11.csv"
    
    print("Simple Selenium Word Counter")
    print("=" * 40)
    print("🎯 Goal: Get accurate word counts without complex filtering")
    
    threshold = input("Word threshold (default 1000): ")
    threshold = int(threshold) if threshold else 1000
    
    results = simple_check(csv_file, threshold=threshold, sample_size=5)

Simple Selenium Word Counter
🎯 Goal: Get accurate word counts without complex filtering


Word threshold (default 1000):  500


📊 Loading data...
✅ Testing 5 papers
📏 Threshold: 500 words

📄 Paper 1/5: PMID 24684821.0
📋 Chemerin levels as predictor of acute coronary events: a case-control study nest...
   🌐 Loading: https://doi.org/10.1016/j.metabol.2014.02.013
   📍 Final URL: https://www.metabolismjournal.com/article/S0026-0495(14)00052-3/abstract
   📊 Raw words: 3068, Filtered: 2816
   📝 Sample: Chemerin levels as predictor of acute coronary events: case–control study nested within the veterans affairs normative aging study Metabolism Clinical...
   🟢 OPEN_ACCESS: 2,816 words

📄 Paper 2/5: PMID 22336131.0
📋 Residential black carbon exposure and circulating markers of systemic inflammati...
   🌐 Loading: https://doi.org/10.1289/ehp.1103982
   📍 Final URL: https://ehp.niehs.nih.gov/doi/10.1289/ehp.1103982
   📊 Raw words: 10272, Filtered: 9452
   📝 Sample: Residential Black Carbon Exposure and Circulating Markers of Systemic Inflammation in Elderly Males: The Normative Aging Study Environmental Health Perspectiv

In [62]:
import pandas as pd
import time
from datetime import datetime
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

def simple_word_count(url, timeout=20):
    """
    Simple word counting with minimal filtering
    """
    
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    driver = None
    try:
        print(f"   🌐 Loading: {url}")
        
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url)
        
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(3)
        
        final_url = driver.current_url
        print(f"   📍 Final URL: {final_url}")
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Remove only the most basic unwanted elements
        for unwanted in soup(["script", "style", "nav", "header", "footer"]):
            unwanted.decompose()
        
        # Get all text
        full_text = soup.get_text(separator=' ', strip=True)
        
        # Very basic cleaning : just normalize whitespace
        full_text = re.sub(r'\s+', ' ', full_text)
        
        # Simple word split and count
        words = full_text.split()
        
        # Minimal filtering : just remove very short words and pure numbers
        filtered_words = [word for word in words if len(word) >= 2 and not word.isdigit()]
        
        word_count = len(filtered_words)
        
        # Show first few words for verification
        sample = ' '.join(filtered_words[:20])
        
        print(f"   📊 Raw words: {len(words)}, Filtered: {word_count}")
        print(f"   📝 Sample: {sample}...")
        
        return word_count, sample
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")
        return 0, f"Error: {str(e)}"
    
    finally:
        if driver:
            driver.quit()

def simple_check(csv_file, threshold=1000, sample_size=5):
    """
    Simple check with minimal complexity
    """
    
    print("📊 Loading data...")
    df = pd.read_csv(csv_file)
    
    # Basic cleaning
    df = df.dropna(subset=['DOI_URL', 'PMID'])
    df = df[df['DOI_URL'].str.startswith('http')]
    
    if sample_size:
        df = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    
    print(f"✅ Testing {len(df)} papers")
    print(f"📏 Threshold: {threshold} words")
    print("=" * 60)
    
    results = []
    
    for i, row in df.iterrows():
        print(f"\n📄 Paper {i+1}/{len(df)}: PMID {row['PMID']}")
        print(f"📋 {row['Title'][:80]}...")
        
        word_count, sample = simple_word_count(row['DOI_URL'])
        
        status = "OPEN_ACCESS" if word_count >= threshold else "RESTRICTED"
        emoji = "🟢" if status == "OPEN_ACCESS" else "🔴"
        
        print(f"   {emoji} {status}: {word_count:,} words")
        
        results.append({
            'PMID': row['PMID'],
            'Title': row['Title'],
            'DOI_URL': row['DOI_URL'],
            'Word_Count': word_count,
            'Status': status,
            'Sample': sample
        })
        
        time.sleep(2)
    
    # Summary
    results_df = pd.DataFrame(results)
    open_count = len(results_df[results_df['Status'] == 'OPEN_ACCESS'])
    restricted_count = len(results_df[results_df['Status'] == 'RESTRICTED'])
    
    print("\n" + "=" * 60)
    print("📊 RESULTS SUMMARY")
    print("=" * 60)
    print(f"🟢 OPEN ACCESS: {open_count} papers")
    print(f"🔴 RESTRICTED: {restricted_count} papers")
    
    if open_count > 0:
        print(f"\n📚 OPEN ACCESS PAPERS:")
        open_papers = results_df[results_df['Status'] == 'OPEN_ACCESS'].sort_values('Word_Count', ascending=False)
        for _, paper in open_papers.iterrows():
            print(f"   • {paper['Word_Count']:,} words - PMID {paper['PMID']}")
    
    # Save results
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_df.to_csv(f"simple_word_check_{timestamp}.csv", index=False)
    
    if open_count > 0:
        open_papers[['PMID', 'Title', 'DOI_URL', 'Word_Count']].to_csv(
            f"OPEN_ACCESS_SIMPLE_{timestamp}.csv", index=False
        )
        print(f"\n💾 Open access papers saved to: OPEN_ACCESS_SIMPLE_{timestamp}.csv")
    
    return results_df

if __name__ == "__main__":
    csv_file = "output_11.csv"
    
    print("Simple Selenium Word Counter")
    print("=" * 40)
    print("🎯 Goal: Get accurate word counts without complex filtering")
    
    print("\nOptions:")
    print("1. Quick test (5 papers)")
    print("2. Full analysis (all papers)")
    
    choice = input("\nChoose 1 or 2: ")
    
    threshold = input("Word threshold (default 500): ")
    threshold = int(threshold) if threshold else 500
    
    if choice == "1":
        results = simple_check(csv_file, threshold=threshold, sample_size=5)
    else:
        print("\n⚠️ This will process all papers and may take 30-60 minutes.")
        confirm = input("Continue with full analysis? (y/n): ")
        if confirm.lower() == 'y':
            results = simple_check(csv_file, threshold=threshold, sample_size=None)
        else:
            print("Analysis cancelled.")
            results = None

Simple Selenium Word Counter
🎯 Goal: Get accurate word counts without complex filtering

Options:
1. Quick test (5 papers)
2. Full analysis (all papers)



Choose 1 or 2:  2
Word threshold (default 500):  1500



⚠️ This will process all papers and may take 30-60 minutes.


Continue with full analysis? (y/n):  Y


📊 Loading data...
✅ Testing 112 papers
📏 Threshold: 1500 words

📄 Paper 1/112: PMID 30414594.0
📋 Accelerated DNA methylation age and the use of antihypertensive medication among...
   🌐 Loading: https://doi.org/10.18632/aging.101626
   📍 Final URL: https://www.aging-us.com/article/101626/text
   📊 Raw words: 8446, Filtered: 7700
   📝 Sample: Accelerated DNA methylation age and the use of antihypertensive medication among older adults Aging Learn about our FREE Post-Publication Promotion...
   🟢 OPEN_ACCESS: 7,700 words

📄 Paper 2/112: PMID 28592514.0
📋 A Western Diet Pattern Is Associated with Higher Concentrations of Blood and Bon...
   🌐 Loading: https://doi.org/10.3945/jn.117.249060
   📍 Final URL: https://www.sciencedirect.com/science/article/pii/S002231662216298X?via%3Dihub
   📊 Raw words: 10467, Filtered: 9373
   📝 Sample: Western Diet Pattern Is Associated with Higher Concentrations of Blood and Bone Lead among Middle-Aged and Elderly Men ScienceDirect JavaScript...
   🟢 OPEN_AC